# 03 · B3 Fine-tuned IndoBERT (5 reported seeds + 3 H3 seeds)

Fine-tunes indobenchmark/indobert-base-p1 (lr 2e-5, batch 16, up to 5 epochs, best epoch chosen on validation macro-F1). Seeds in `reported_seeds` go into the paper; seeds in `h3_seeds` are used **only** to define hard-case subset H3, so H3 does not favour other methods by construction. Runtime: Colab T4, roughly 10–15 min per SmSA seed.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
import json, time, numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, set_seed)
from sklearn.metrics import f1_score
from mad_indo import stats
bc = cfg["baselines"]["indobert"]
tok = AutoTokenizer.from_pretrained(bc["model"])
out = RESULTS / "baselines"; out.mkdir(exist_ok=True)

def run_seed(task, seed):
    L = cfg["tasks"][task]["labels"]; l2i = {l: i for i, l in enumerate(L)}
    def ds(df):
        d = Dataset.from_pandas(df.assign(labels=df.label.map(l2i))[["text", "labels"]])
        return d.map(lambda b: tok(b["text"], truncation=True, max_length=bc["max_length"]), batched=True)
    tr, va = data.load_split(task, "train"), data.load_split(task, "valid")
    te = data.load_split(task, "test", limit=LIMIT)
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(bc["model"], num_labels=len(L))
    args = TrainingArguments(output_dir=f"/tmp/b3_{task}_{seed}", learning_rate=bc["learning_rate"],
        per_device_train_batch_size=bc["batch_size"], per_device_eval_batch_size=64,
        num_train_epochs=bc["max_epochs"], eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1", save_total_limit=1,
        seed=seed, fp16=torch.cuda.is_available(), report_to=[], logging_steps=50)
    metric = lambda p: {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}
    trainer = Trainer(model=model, args=args, train_dataset=ds(tr), eval_dataset=ds(va),
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=metric)
    t0 = time.perf_counter(); trainer.train(); train_s = time.perf_counter() - t0
    t1 = time.perf_counter(); logits = trainer.predict(ds(te)).predictions; infer_s = time.perf_counter() - t1
    pred = [L[i] for i in logits.argmax(-1)]
    pd.DataFrame({"id": te.id, "gold": te.label, "pred": pred}).to_csv(out / f"B3_indobert_{task}_seed{seed}.csv", index=False)
    json.dump({"seed": seed, "train_s": train_s, "infer_s_per_1000": 1000 * infer_s / len(te)},
              open(out / f"B3_indobert_{task}_seed{seed}.json", "w"), indent=2)
    return stats.metrics(te.label, pred, L)

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [3]:
summary = []
for task in ["smsa", "emot"]:
    for seed in bc["reported_seeds"] + bc["h3_seeds"]:
        if (out / f"B3_indobert_{task}_seed{seed}.csv").exists():
            print("skip (done)", task, seed); continue
        m = run_seed(task, seed)
        summary.append({"task": task, "seed": seed, "role": "reported" if seed in bc["reported_seeds"] else "H3-only", **m})
        print(task, seed, round(m["macro_f1"], 4))
pd.DataFrame(summary).round(4)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (195 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/440 [00:00<?, ? examples/s]

emot 777 0.7385


,task,seed,role,accuracy,macro_f1,unknown_rate,f1_positive,recall_positive,f1_neutral,recall_neutral,...,f1_sadness,recall_sadness,f1_anger,recall_anger,f1_love,recall_love,f1_fear,recall_fear,f1_happy,recall_happy
0,smsa,13,reported,0.9160,0.8872,0.0,0.9362,0.9519,0.7838,0.6591,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,smsa,21,reported,0.9180,0.8968,0.0,0.9327,0.8990,0.8092,0.7955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,smsa,42,reported,0.9140,0.8798,0.0,0.9393,0.9663,0.7586,0.6250,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,smsa,87,reported,0.9080,0.8754,0.0,0.9153,0.9615,0.7568,0.6364,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,smsa,100,reported,0.9260,0.9075,0.0,0.9363,0.9183,0.8333,0.7955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,smsa,7,H3-only,0.9200,0.8939,0.0,0.9384,0.9519,0.7974,0.6932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,smsa,77,H3-only,0.9200,0.9037,0.0,0.9253,0.9231,0.8383,0.7955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,smsa,777,H3-only,0.9260,0.9055,0.0,0.9403,0.9471,0.8280,0.7386,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,emot,13,reported,0.7364,0.7422,0.0,NaN,NaN,NaN,NaN,...,0.6178,0.59,0.7712,0.8273,0.8000,0.7812,0.7805,0.7385,0.7415,0.7525
9,emot,21,reported,0.7341,0.7459,0.0,NaN,NaN,NaN,NaN,...,0.6520,0.74,0.7320,0.6455,0.7667,0.7188,0.8264,0.7692,0.7523,0.8119
